# H3K27me3 within Keren: train on Keren patients, test on held-out Keren patients

**Why.** In `epi-relations` v3, a Jackson-trained LightGBM predicted Keren's measured H3K27me3 poorly: the partial r
beyond nuclear content was 0.12. The same inputs reached 0.61 within Jackson. This kernel asks whether Keren's mark is
predictable from Keren's own channels.
- If it is, the 0.12 is a Jackson → Keren transfer failure.
- If it is not, H3K27me3 in Keren is not linked to these markers, and no Jackson model could transfer it.

**Split.** 5-fold cross-validation by patient. Keren has one image per patient, so held-out patients = held-out images,
and no cell of a test image is seen in training. Predictions are out-of-fold for all 40 patients.

**Models** (LightGBM, the same settings as `epi-relations`; features rank-normalised within Keren):
- **nuclear:** dsDNA and log area.
- **shared 9:** dsDNA, Ki-67, CD3, CD20, CD45, CD68, SMA, panCK and log area. These are the inputs of the Jackson → Keren test.
- **all channels:** every Keren channel except H3K27me3 and H3K9ac (34 + log area). The upper bound for this panel.
- **all, no mass neighbours:** all channels except panCK (Yb171) and phospho-S6 (Yb173), which are ±1 Da from H3K27me3 (Yb172),
  and CD68 (Gd156), whose oxide lands at 172. This guards against isotope impurity or oxide crosstalk. It was added after a
  local smoke run showed phospho-S6 carrying most of the model's gain.
- **all + H3K9ac:** adds the other histone mark (Er170, 2 Da away).

**Scores** (as in `epi-relations`; values arcsinh(x / 1); 1000× patient bootstrap):
- within-image r;
- partial r beyond nuclear content: within images, given dsDNA and log area, on normal scores;
- patient Spearman: epithelial-cell means; patients with ≥ 50 epithelial cells.

**Reference points** from `epi-relations` v3, with the shared-9 inputs:
- Jackson-trained, scored in Jackson test patients: partial r 0.61.
- Jackson-trained, scored in Keren: partial r 0.12.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings
import numpy as np, pandas as pd
from scipy import stats
from scipy.special import ndtri
from sklearn.model_selection import GroupKFold
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261022
COFACTOR = 1.0
N_FOLD = 5
N_BOOT = 100 if SMOKE else 1000
MIN_PAT_CELLS = 50
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
TARGET, OTHER_EPI = "h3k27me3", "h3k9ac"
SHARED = ["ds_dna", "ki67", "cd3", "cd20", "cd45", "cd68", "sma", "pan_keratin"]
MASS_NEIGHBOURS = ["pan_keratin", "phospho_s6", "cd68"]   # Yb171 and Yb173 (±1 Da of H3K27me3 Yb172); Gd156 (oxide, +16 = 172)
REF = {"Jackson-trained, Jackson test (epi-relations v3)": 0.607, "Jackson-trained, Keren (epi-relations v3)": 0.117}
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def ci(v):
    v = np.asarray(v, np.float64)
    v = v[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) else (np.nan, np.nan)

def nscore(v):
    v = np.asarray(v, np.float64)
    return ndtri((stats.rankdata(v) - 0.5) / len(v))

def demean(X, img):
    X = np.asarray(X, np.float64)
    k = int(img.max()) + 1
    cnt = np.maximum(np.bincount(img, minlength=k), 1).astype(np.float64)
    if X.ndim == 1:
        return X - (np.bincount(img, X, k) / cnt)[img]
    return X - np.column_stack([np.bincount(img, X[:, j], k) / cnt for j in range(X.shape[1])])[img]

def rank_norm(X):
    return np.column_stack([(stats.rankdata(X[:, j]) - 0.5) / len(X) for j in range(X.shape[1])]).astype(np.float32)

## Data

In [ ]:
p = [f for f in glob.glob(f"{IN}/**/cells.parquet", recursive=True) if "keren" in f.lower()]
assert len(p) == 1, p
try:
    c = pd.read_parquet(p[0])
except Exception:            # local pyarrow cannot read some Kaggle-written files; duckdb can
    import duckdb
    c = duckdb.query(f"select * from read_parquet('{p[0]}')").df()
c = c.sort_values(["image_name", "object_number"]).reset_index(drop=True)
CH = [k for k in c.columns if k not in ("ds_dna",) and not k.startswith(("eroded__", "published__")) and
      c[k].dtype.kind == "f" and k not in ("area", "area_um2", "area_eroded", "x", "y", "x_um", "y_um")]
CH = ["ds_dna"] + CH
assert TARGET in CH and OTHER_EPI in CH and all(s in CH for s in SHARED), CH
print(len(CH), "channels:", CH)
A = {k: np.arcsinh(c[k].values.astype(np.float64) / COFACTOR) for k in CH}
log_area = np.log(np.maximum(c.area.values.astype(np.float64), 1))
img = pd.factorize(c.image_name, sort=True)[0].astype(np.int64)
pats, pcode = np.unique(c.patient_id.astype(str).values, return_inverse=True)
broad = c.broad_class.values.astype(str)
n, n_pat = len(c), len(pats)
print(f"{n:,} cells, {n_pat} patients, {img.max() + 1} images")
rng = np.random.default_rng(SEED + 7)
W = rng.multinomial(n_pat, np.full(n_pat, 1 / n_pat), size=N_BOOT).astype(np.float64)

## Out-of-fold predictions (5 folds by patient)

In [ ]:
NON_EPI = [k for k in CH if k not in (TARGET, OTHER_EPI)]
SPECS = {"nuclear": ["ds_dna"], "shared 9": SHARED, "all channels": NON_EPI,
         "all, no mass neighbours": [k for k in NON_EPI if k not in MASS_NEIGHBOURS], "all + H3K9ac": NON_EPI + [OTHER_EPI]}
y = A[TARGET]
folds = list(GroupKFold(N_FOLD).split(np.zeros(n), groups=pcode))
PRED = {}
for name, chans in SPECS.items():
    X = rank_norm(np.column_stack([A[k] for k in chans] + [log_area]))
    o = np.full(n, np.nan)
    for tr, te in folds:
        o[te] = lgb.LGBMRegressor(random_state=SEED, **LGB).fit(X[tr], y[tr]).predict(X[te])
    PRED[name] = o
    tick(f"{name}: {X.shape[1]} features")
assert all(np.isfinite(v).all() for v in PRED.values())

## Scores

In [ ]:
N = demean(np.column_stack([nscore(A["ds_dna"]), nscore(log_area)]), img)
Q, _ = np.linalg.qr(N)

def pooled_r(a_, b_):
    sxy, sxx, syy = (np.bincount(pcode, w, n_pat) for w in (a_ * b_, a_ * a_, b_ * b_))
    return float(sxy.sum() / np.sqrt(sxx.sum() * syy.sum())), (W @ sxy) / np.sqrt((W @ sxx) * (W @ syy))

def adjusted(x):
    r = demean(x, img)
    return r - Q @ (Q.T @ r)

def epi_mean(x):
    e = broad == "epithelial"
    cnt = np.bincount(pcode[e], minlength=n_pat)
    return np.where(cnt >= MIN_PAT_CELLS, np.bincount(pcode[e], x[e], n_pat) / np.maximum(cnt, 1), np.nan)

def spearman_boot(x, yv):
    ok = np.isfinite(x) & np.isfinite(yv)
    x, yv = x[ok], yv[ok]
    BI = np.random.default_rng(SEED + 3).integers(0, len(x), (N_BOOT, len(x)))
    rx, ry = stats.rankdata(x[BI], axis=1), stats.rankdata(yv[BI], axis=1)
    rx -= rx.mean(1, keepdims=True)
    ry -= ry.mean(1, keepdims=True)
    return float(stats.spearmanr(x, yv).correlation), (rx * ry).sum(1) / np.sqrt((rx ** 2).sum(1) * (ry ** 2).sum(1)), int(ok.sum())

ey = nscore(y)
rows = []
for name, p_ in PRED.items():
    r, rb = pooled_r(demean(y, img), demean(p_, img))
    pr, prb = pooled_r(adjusted(ey), adjusted(nscore(p_)))
    sp, spb, npat = spearman_boot(epi_mean(nscore(p_)), epi_mean(ey))
    rows.append(dict(model=name, n_features=len(SPECS[name]) + 1, r_within=r, r_lo=ci(rb)[0], r_hi=ci(rb)[1],
                     partial_beyond_nuclear=pr, partial_lo=ci(prb)[0], partial_hi=ci(prb)[1],
                     patient_spearman=sp, patient_lo=ci(spb)[0], patient_hi=ci(spb)[1], n_patients=npat))
S = pd.DataFrame(rows)
S.to_csv(f"{OUT}/keren_within_scores.csv", index=False)
print(S.round(3).to_string(index=False))

# which channels carry the signal (all-channels model, gain importance on all cells; descriptive only)
X = rank_norm(np.column_stack([A[k] for k in NON_EPI] + [log_area]))
m = lgb.LGBMRegressor(random_state=SEED, importance_type="gain", **LGB).fit(X, y)
imp = pd.Series(m.feature_importances_, index=NON_EPI + ["log_area"]).sort_values(ascending=False)
imp = (imp / imp.sum()).rename("gain_share")
imp.to_csv(f"{OUT}/keren_within_importance.csv")
print(imp.head(12).round(3).to_string())
# partial r of the measured mark with each channel, beyond nuclear content (within images, all cells)
pc = {k: pooled_r(adjusted(ey), adjusted(nscore(A[k])))[0] for k in NON_EPI + [OTHER_EPI] if k != "ds_dna"}
pc = pd.Series(pc).sort_values(key=np.abs, ascending=False).rename("partial_r_with_h3k27me3")
pc.to_csv(f"{OUT}/keren_within_channel_partials.csv")
print(pc.head(12).round(3).to_string())

## Figure and summary

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.8))
d = S.set_index("model")
x = np.arange(len(d))
ax.bar(x, d.partial_beyond_nuclear, width=0.65, color="#1f4e9c")
ax.errorbar(x, d.partial_beyond_nuclear, yerr=[d.partial_beyond_nuclear - d.partial_lo, d.partial_hi - d.partial_beyond_nuclear],
            fmt="none", color="black", capsize=3, lw=1)
for (lab, v), ls in zip(REF.items(), ("--", ":")):
    ax.axhline(v, color="#c0392b", ls=ls, lw=1.2)
    ax.text(len(d) - 0.45, v + 0.01, lab.replace(" (epi-relations v3)", ""), ha="right", va="bottom", fontsize=7, color="#c0392b")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"Keren-trained,\n{k}" for k in d.index], fontsize=8)
ax.set_ylabel("H3K27me3: partial r beyond nuclear content\n(Keren, held-out patients)", fontsize=9)
ax.set_ylim(min(0, float(d.partial_lo.min())) - 0.05, max(0.75, float(d.partial_hi.max()) + 0.05))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{OUT}/fig_keren_within.png", dpi=200)
plt.close(fig)
json.dump(dict(smoke=SMOKE, cells=n, patients=n_pat, channels=len(CH), reference=REF, scores=S.round(4).to_dict(orient="records"),
               top_importance=imp.head(10).round(4).to_dict(), top_partials=pc.head(10).round(4).to_dict(),
               minutes=(time.time() - T0) / 60), open(f"{OUT}/summary.json", "w"), indent=1)
tick("done")